# EDA Reproducible — Superstore Dataset
## Pipeline completo: Datos crudos → Warehouse → Modelos entrenados

**Dataset**: Superstore Sales (~9,994 filas, 21 columnas)

**Preguntas**:
1. **Regresión**: ¿Cuánto profit generará una orden?
2. **Clasificación**: ¿Una orden será rentable (profit > 0)?

In [1]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json, joblib, warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Rutas del proyecto
PROJECT_ROOT = Path(os.path.abspath('')).parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'superstore_crudo.csv'
MODELS_DIR = PROJECT_ROOT / 'saved_models'
FIGURES_DIR = Path('figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Agregar backend al path para usar database.py
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Proyecto: {PROJECT_ROOT}')
print(f'Dataset:  {DATA_RAW}')
print(f'Modelos:  {MODELS_DIR}')

Proyecto: D:\PROYECTO_FULLSTACK_DATA
Dataset:  D:\PROYECTO_FULLSTACK_DATA\data\raw\superstore_crudo.csv
Modelos:  D:\PROYECTO_FULLSTACK_DATA\saved_models


## 1. Carga de Datos e Inspección Inicial

In [2]:
df = pd.read_csv(DATA_RAW, encoding='latin-1')
print(f'Shape: {df.shape}')
print(f'\nColumnas: {df.columns.tolist()}')
print(f'\nTipos de datos:')
print(df.dtypes)
print(f'\nNulos por columna:')
print(df.isnull().sum())
print(f'\nDuplicados: {df.duplicated().sum()}')
df.head()

Shape: (9994, 21)

Columnas: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Tipos de datos:
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

Nulos por columna:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Se

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
# Limpieza de nombres de columnas
df.columns = [c.strip().replace(' ', '_') for c in df.columns]
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='mixed')
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'], format='mixed')
df.describe()

,Row_ID,Order_Date,Ship_Date,Postal_Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994,9994,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,2016-04-30 00:07:12.259355,2016-05-03 23:06:58.571142,55190.379428,229.858001,3.789574,0.156203,28.656896
min,1.000000,2014-01-03 00:00:00,2014-01-07 00:00:00,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,2015-05-23 00:00:00,2015-05-27 00:00:00,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,2016-06-26 00:00:00,2016-06-29 00:00:00,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,2017-05-14 00:00:00,2017-05-18 00:00:00,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,2017-12-30 00:00:00,2018-01-05 00:00:00,99301.000000,22638.480000,14.000000,0.800000,8399.976000
std,2885.163629,NaN,NaN,32063.693350,623.245101,2.225110,0.206452,234.260108


## 2. Análisis Exploratorio de Datos (EDA)

In [4]:
# 2.1 Distribuciones de variables numéricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), ['Sales', 'Profit', 'Quantity', 'Discount']):
    ax.hist(df[col], bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribución de {col}')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Media: {df[col].mean():.2f}')
    ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'distribuciones.png', bbox_inches='tight')
plt.show()

In [5]:
# 2.2 Ventas y Profit por Categoría
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cat_stats = df.groupby('Category').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
axes[0].bar(cat_stats['Category'], cat_stats['Sales'], color=['#2196F3', '#FF9800', '#4CAF50'])
axes[0].set_title('Ventas Totales por Categoría')
axes[0].set_ylabel('Ventas (USD)')
axes[1].bar(cat_stats['Category'], cat_stats['Profit'], color=['#2196F3', '#FF9800', '#4CAF50'])
axes[1].set_title('Profit Total por Categoría')
axes[1].set_ylabel('Profit (USD)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'categoria_ventas_profit.png', bbox_inches='tight')
plt.show()

In [6]:
# 2.3 Profit por Sub-Categoría (identificar las que pierden dinero)
sub_profit = df.groupby('Sub-Category')['Profit'].sum().sort_values()
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in sub_profit.values]
fig, ax = plt.subplots(figsize=(12, 8))
sub_profit.plot(kind='barh', color=colors, ax=ax)
ax.set_title('Profit Total por Sub-Categoría')
ax.set_xlabel('Profit (USD)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'subcategoria_profit.png', bbox_inches='tight')
plt.show()
print('Sub-categorías con pérdida:', sub_profit[sub_profit < 0].index.tolist())

Sub-categorías con pérdida: ['Tables', 'Bookcases', 'Supplies']


In [7]:
# 2.4 Ventas por Región
region_stats = df.groupby('Region').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(region_stats))
ax.bar(x - 0.2, region_stats['Sales'], 0.4, label='Sales', color='#3498db')
ax.bar(x + 0.2, region_stats['Profit'], 0.4, label='Profit', color='#e67e22')
ax.set_xticks(x)
ax.set_xticklabels(region_stats['Region'])
ax.legend()
ax.set_title('Ventas y Profit por Región')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'region_ventas.png', bbox_inches='tight')
plt.show()

In [8]:
# 2.5 Tendencia temporal
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
monthly = df.groupby(['Year', 'Month']).agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
monthly['Period'] = monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str).str.zfill(2)
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly['Period'], monthly['Sales'], marker='o', label='Sales', linewidth=2)
ax.plot(monthly['Period'], monthly['Profit'], marker='s', label='Profit', linewidth=2)
ax.set_title('Tendencia Mensual de Ventas y Profit')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'tendencia_mensual.png', bbox_inches='tight')
plt.show()

In [9]:
# 2.6 Correlación entre variables numéricas
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='RdYlBu_r', center=0, ax=ax, fmt='.3f')
ax.set_title('Matriz de Correlación')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlacion.png', bbox_inches='tight')
plt.show()
print('\nInsight clave: Discount tiene correlación NEGATIVA con Profit (-0.22)')


Insight clave: Discount tiene correlación NEGATIVA con Profit (-0.22)


In [10]:
# 2.7 Descuento vs Profit (scatter)
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(df['Discount'], df['Profit'], alpha=0.3, c=df['Sales'], cmap='viridis', s=10)
plt.colorbar(scatter, label='Sales')
ax.set_xlabel('Discount')
ax.set_ylabel('Profit')
ax.set_title('Descuento vs Profit (color = Sales)')
ax.axhline(0, color='red', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'discount_vs_profit.png', bbox_inches='tight')
plt.show()

# Distribución de la variable target de clasificación
df['is_profitable'] = (df['Profit'] > 0).astype(int)
print(f'\nDistribución de rentabilidad:')
print(df['is_profitable'].value_counts(normalize=True).round(4) * 100)


Distribución de rentabilidad:
is_profitable
1    80.63
0    19.37
Name: proportion, dtype: float64


## 3. Preprocesamiento Parametrizado

**Decisiones justificadas:**
- **StandardScaler** en numéricas: los modelos lineales requieren features escaladas
- **OneHotEncoder** en categóricas: preserva información sin asumir ordinalidad
- **Train/test split ANTES de fit**: evita fuga de datos
- **Stratified split para clasificación**: preserva proporción de clases

In [11]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SKPipeline

# Feature engineering
df['Shipping_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days

# Definir features
numeric_features = ['Sales', 'Quantity', 'Discount', 'Shipping_Days']
categorical_features = ['Ship_Mode', 'Segment', 'Category', 'Sub_Category', 'Region']

# Renombrar Sub-Category para consistencia
df = df.rename(columns={'Sub-Category': 'Sub_Category'})

# Features y targets
feature_cols = numeric_features + categorical_features
X = df[feature_cols].copy()
y_regression = df['Profit'].copy()
y_classification = df['is_profitable'].copy()

print(f'Features shape: {X.shape}')
print(f'Numeric: {numeric_features}')
print(f'Categorical: {categorical_features}')
print(f'\nTarget regresión (Profit) - Media: {y_regression.mean():.2f}, Std: {y_regression.std():.2f}')
print(f'Target clasificación (is_profitable) - Distribución: {y_classification.value_counts().to_dict()}')

Features shape: (9994, 9)
Numeric: ['Sales', 'Quantity', 'Discount', 'Shipping_Days']
Categorical: ['Ship_Mode', 'Segment', 'Category', 'Sub_Category', 'Region']

Target regresión (Profit) - Media: 28.66, Std: 234.26
Target clasificación (is_profitable) - Distribución: {1: 8058, 0: 1936}


In [12]:
# Preprocesador (se ajusta SOLO en train)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='drop'
)

# Split 80/20, estratificado para clasificación
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_regression, y_classification, test_size=0.2, random_state=42, stratify=y_classification
)

print(f'Train: {X_train.shape[0]} filas  |  Test: {X_test.shape[0]} filas')
print(f'Proporción rentable (train): {y_clf_train.mean():.4f}')
print(f'Proporción rentable (test):  {y_clf_test.mean():.4f}')

# Ajustar preprocesador SOLO en train
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
print(f'\nFeatures después de preprocesamiento: {X_train_proc.shape[1]}')

Train: 7995 filas  |  Test: 1999 filas


Proporción rentable (train): 0.8063
Proporción rentable (test):  0.8064

Features después de preprocesamiento: 35


## 4. Regresión — Predicción de Profit

Comparamos 3 modelos. Métricas: **MAE, RMSE, R²**
- MAE: error promedio en dólares (interpretable)
- RMSE: penaliza errores grandes (importante para outliers de profit)
- R²: varianza explicada

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

reg_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
}

reg_results = []
best_reg_score = -np.inf
best_reg_name = ''
best_reg_model = None

for name, model in reg_models.items():
    model.fit(X_train_proc, y_reg_train)
    y_pred = model.predict(X_test_proc)
    
    mae = mean_absolute_error(y_reg_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred))
    r2 = r2_score(y_reg_test, y_pred)
    
    reg_results.append({'Model': name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R²': round(r2, 4)})
    
    if r2 > best_reg_score:
        best_reg_score = r2
        best_reg_name = name
        best_reg_model = model

reg_df = pd.DataFrame(reg_results)
print('=== Comparación de Modelos de Regresión ===')
print(reg_df.to_string(index=False))
print(f'\n✅ Mejor modelo: {best_reg_name} (R² = {best_reg_score:.4f})')

=== Comparación de Modelos de Regresión ===
            Model   MAE   RMSE     R²
Linear Regression 60.49 243.30 0.0932
    Random Forest 22.10 168.14 0.5669
Gradient Boosting 22.06 154.52 0.6343

✅ Mejor modelo: Gradient Boosting (R² = 0.6343)


In [14]:
# Visualización: Predicción vs Real
y_pred_best = best_reg_model.predict(X_test_proc)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(y_reg_test, y_pred_best, alpha=0.3, s=10)
axes[0].plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Profit Real')
axes[0].set_ylabel('Profit Predicho')
axes[0].set_title(f'{best_reg_name}: Predicho vs Real')

residuals = y_reg_test - y_pred_best
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribución de Residuales')
axes[1].axvline(0, color='red', linestyle='--')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'regresion_resultados.png', bbox_inches='tight')
plt.show()

## 5. Clasificación — ¿Es Rentable?

Comparamos 4 algoritmos del curso. **Métricas: F1 (macro), ROC-AUC, Precision, Recall**

No usamos accuracy sola porque la clase positiva es ~80% — un modelo naive alcanzaría 80% accuracy sin aprender nada. F1-macro y ROC-AUC son más informativos.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (f1_score, roc_auc_score, precision_score, 
                             recall_score, classification_report, confusion_matrix)

clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-NN (k=7)': KNeighborsClassifier(n_neighbors=7),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Naive Bayes': GaussianNB()
}

clf_results = []
best_clf_score = -np.inf
best_clf_name = ''
best_clf_model = None

for name, model in clf_models.items():
    model.fit(X_train_proc, y_clf_train)
    y_pred = model.predict(X_test_proc)
    y_prob = model.predict_proba(X_test_proc)[:, 1]
    
    f1 = f1_score(y_clf_test, y_pred, average='macro')
    roc = roc_auc_score(y_clf_test, y_prob)
    prec = precision_score(y_clf_test, y_pred, average='macro')
    rec = recall_score(y_clf_test, y_pred, average='macro')
    
    clf_results.append({'Model': name, 'F1-macro': round(f1, 4), 'ROC-AUC': round(roc, 4),
                        'Precision': round(prec, 4), 'Recall': round(rec, 4)})
    
    if f1 > best_clf_score:
        best_clf_score = f1
        best_clf_name = name
        best_clf_model = model

clf_df = pd.DataFrame(clf_results)
print('=== Comparación de Modelos de Clasificación ===')
print(clf_df.to_string(index=False))
print(f'\n✅ Mejor modelo: {best_clf_name} (F1-macro = {best_clf_score:.4f})')

=== Comparación de Modelos de Clasificación ===
              Model  F1-macro  ROC-AUC  Precision  Recall
Logistic Regression    0.8989   0.9812     0.9267  0.8767
         K-NN (k=7)    0.8889   0.9553     0.9234  0.8628
      Decision Tree    0.8992   0.9542     0.9090  0.8902
        Naive Bayes    0.4716   0.9014     0.6348  0.6749

✅ Mejor modelo: Decision Tree (F1-macro = 0.8992)


In [16]:
# Classification report y confusion matrix del mejor modelo
y_pred_best_clf = best_clf_model.predict(X_test_proc)
print(f'=== Classification Report: {best_clf_name} ===')
print(classification_report(y_clf_test, y_pred_best_clf, target_names=['No Rentable', 'Rentable']))

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_clf_test, y_pred_best_clf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Rentable', 'Rentable'],
            yticklabels=['No Rentable', 'Rentable'], ax=ax)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title(f'Confusion Matrix — {best_clf_name}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', bbox_inches='tight')
plt.show()

=== Classification Report: Decision Tree ===
              precision    recall  f1-score   support

 No Rentable       0.86      0.81      0.84       387
    Rentable       0.96      0.97      0.96      1612

    accuracy                           0.94      1999
   macro avg       0.91      0.89      0.90      1999
weighted avg       0.94      0.94      0.94      1999



## 6. Guardar Modelos y Metadatos

In [17]:
# Guardar modelos entrenados
joblib.dump(best_reg_model, MODELS_DIR / 'regression_model.joblib')
joblib.dump(best_clf_model, MODELS_DIR / 'classification_model.joblib')
joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.joblib')

# Guardar metadatos
best_reg_metrics = reg_df[reg_df['Model'] == best_reg_name].iloc[0].to_dict()
best_clf_metrics = clf_df[clf_df['Model'] == best_clf_name].iloc[0].to_dict()

metadata = {
    'regression': {
        'model_name': best_reg_name,
        'metrics': {k: v for k, v in best_reg_metrics.items() if k != 'Model'}
    },
    'classification': {
        'model_name': best_clf_name,
        'metrics': {k: v for k, v in best_clf_metrics.items() if k != 'Model'}
    },
    'training_date': datetime.now().isoformat(),
    'feature_names': feature_cols,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'train_size': len(X_train),
    'test_size': len(X_test),
    'all_regression_results': reg_results,
    'all_classification_results': clf_results
}

with open(MODELS_DIR / 'model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('✅ Modelos guardados en saved_models/')
print(f'   - regression_model.joblib ({best_reg_name})')
print(f'   - classification_model.joblib ({best_clf_name})')
print(f'   - preprocessor.joblib')
print(f'   - model_metadata.json')

✅ Modelos guardados en saved_models/
   - regression_model.joblib (Gradient Boosting)
   - classification_model.joblib (Decision Tree)
   - preprocessor.joblib
   - model_metadata.json


## 7. Creación del Warehouse DuckDB

In [18]:
from backend.app.database import init_warehouse, DB_PATH

init_warehouse(str(DATA_RAW))
print(f'\nWarehouse guardado en: {DB_PATH}')

✅ Warehouse creado en D:\PROYECTO_FULLSTACK_DATA\data\superstore_warehouse.duckdb
   fact_sales: 9994 filas
   dim_customer: 793 filas
   dim_product: 1894 filas
   dim_geography: 632 filas
   dim_date: 1434 filas

Warehouse guardado en: D:\PROYECTO_FULLSTACK_DATA\data\superstore_warehouse.duckdb


## 8. Limitaciones y Trabajo Futuro

### Limitaciones del modelo:
- **Tamaño del dataset**: ~10K filas es modesto; más datos mejorarían la generalización
- **Variables faltantes**: no tenemos costo del producto, lo que limita el modelado de profit
- **Outliers de profit**: valores extremos (-6600 a +8400) distorsionan la regresión
- **Datos geográficos limitados**: solo USA, un solo país no permite generalizar

### Qué haríamos diferente:
- Feature engineering más sofisticado (RFM del cliente, estacionalidad)
- Probar modelos ensemble (XGBoost, LightGBM)
- Cross-validation más exhaustivo con búsqueda bayesiana de hiperparámetros